In [0]:
df = spark.table("thekitchen.av.five_year_treasury")

display(df.orderBy("timestamp", ascending=False))

In [0]:
import requests


# API key for Alpha Vantage
api_key = "CCQUOAYL7LHVG0Z1"

# Define the base URL and parameters
function = "TREASURY_YIELD"
interval = "daily"
maturity = "5year"
datatype = "csv"

# Construct the URL
url = f"https://www.alphavantage.co/query?function={function}&interval={interval}&maturity={maturity}&datatype={datatype}&apikey={api_key}"

# Make the API request
r = requests.get(url)
data = r.text

# Print the data
print("\n".join(data.splitlines()[:5]))

In [0]:
import pandas as pd
import io
from pyspark.sql import functions as F

# Read CSV
df = pd.read_csv(io.StringIO(data))

# Replace "." with None in 'value' column
df["value"] = df["value"].replace(".", None)

# Convert 'timestamp' column to datetime
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce").dt.date

# Convert 'value' column to float, coerce errors to NaN
df["value"] = pd.to_numeric(df["value"], errors="coerce")

# Pandas → Spark
spark_df = spark.createDataFrame(df)

# Add audit columns in Spark
spark_df = spark_df.withColumn(
    "CreatedAt",
    F.date_trunc(
        "second", F.from_utc_timestamp(F.current_timestamp(), "Europe/Berlin")
    ),
).withColumn("UpdatedAt", F.lit(None).cast("timestamp"))

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import current_timestamp, from_utc_timestamp, date_trunc, lit

target_table = "thekitchen.av.five_year_treasury"
delta_table = DeltaTable.forName(spark, target_table)

cet_timestamp = date_trunc(
    "second", from_utc_timestamp(current_timestamp(), "Europe/Berlin")
)

try:
    delta_table.alias("target").merge(
        source=spark_df.alias("source"), condition="target.timestamp = source.timestamp"
    ).whenMatchedUpdate(
        set={"value": "source.value", "UpdatedAt": cet_timestamp}
    ).whenNotMatchedInsert(
        values={
            "timestamp": "source.timestamp",
            "value": "source.value",
            "CreatedAt": cet_timestamp,
            "UpdatedAt": lit(None).cast("timestamp"),
        }
    ).execute()
    print("Merge operation completed successfully.")
except Exception as e:
    print(f"Merge operation failed: {e}")

In [0]:
df_updated = spark.table("thekitchen.av.five_year_treasury").orderBy("timestamp", ascending=False)
display(df_updated)